# 01 · Python for GIS: the basics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oechenique/python-gis-basics/blob/main/notebooks/01_Intro_To_Python_Gis.ipynb)

Plain Python (no libraries) applied to coordinates: variables, lists and dictionaries, `if`, `for` and functions.
It ends with the first trap of spatial work: **a degree is not a distance**. Notebook [02](02_Working_With_Data.ipynb) moves to pandas and [03](03_Intro_To_Geodata.ipynb) to GeoPandas.

## 1. Variables and types

A point on Earth is two numbers. GIS libraries (shapely, GeoPandas, GeoJSON) use **(x, y) = (longitude, latitude)**; Google Maps and most people say "latitude, longitude". Pick one order and write it down.

In [1]:
city = "Buenos Aires"     # str
population_rank = 1       # int
latitude = -34.6037       # float, degrees north (negative = south)
longitude = -58.3816      # float, degrees east (negative = west)
is_capital = True         # bool

point = (longitude, latitude)  # tuple in (x, y) order
print(type(city).__name__, type(latitude).__name__, type(is_capital).__name__)
print(f"{city}: x={point[0]}, y={point[1]}")

str float bool
Buenos Aires: x=-58.3816, y=-34.6037


## 2. Lists and dictionaries

A list keeps order; a dictionary looks things up by name.

In [2]:
cities = {
    "Buenos Aires": (-58.3816, -34.6037),
    "Córdoba": (-64.1888, -31.4201),
    "Mendoza": (-68.8458, -32.8895),
    "Salta": (-65.4232, -24.7821),
    "Ushuaia": (-68.3030, -54.8019),
}
names = list(cities)
print(len(names), "cities:", names)
print("Córdoba is at", cities["Córdoba"])

5 cities: ['Buenos Aires', 'Córdoba', 'Mendoza', 'Salta', 'Ushuaia']
Córdoba is at (-64.1888, -31.4201)


## 3. Decisions (`if`) and loops (`for`)

In [3]:
def latitude_band(lat):
    """Rough climate band from latitude (south is negative)."""
    if lat > -23.44:
        return "tropical"
    elif lat > -40:
        return "temperate (north)"
    else:
        return "temperate/cold (south)"


for name, (lon, lat) in cities.items():
    print(f"{name:<13} lat {lat:>8.4f}  ->  {latitude_band(lat)}")

Buenos Aires  lat -34.6037  ->  temperate (north)
Córdoba       lat -31.4201  ->  temperate (north)
Mendoza       lat -32.8895  ->  temperate (north)
Salta         lat -24.7821  ->  temperate (north)
Ushuaia       lat -54.8019  ->  temperate/cold (south)


## 4. Functions: two ways to measure a distance

The intuitive one is Pythagoras on the coordinates. It returns a number, but in **degrees**, and a degree of longitude shrinks towards the poles.
The [haversine formula](https://en.wikipedia.org/wiki/Haversine_formula) treats the Earth as a sphere and returns kilometres.

In [4]:
import math

EARTH_RADIUS_KM = 6371.0


def naive_distance(p1, p2):
    """Pythagoras on (lon, lat). The result is in degrees, not in km."""
    return math.hypot(p2[0] - p1[0], p2[1] - p1[1])


def haversine_km(p1, p2):
    """Great-circle distance in km between two (lon, lat) points on a sphere."""
    lon1, lat1, lon2, lat2 = map(math.radians, (*p1, *p2))
    a = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    return 2 * EARTH_RADIUS_KM * math.asin(math.sqrt(a))


ba = cities["Buenos Aires"]
for name in ["Córdoba", "Mendoza", "Ushuaia"]:
    deg = naive_distance(ba, cities[name])
    km = haversine_km(ba, cities[name])
    print(f"Buenos Aires -> {name:<8} naive {deg:5.2f} degrees | haversine {km:6.0f} km | km per 'degree' {km / deg:5.1f}")

Buenos Aires -> Córdoba  naive  6.62 degrees | haversine    647 km | km per 'degree'  97.7
Buenos Aires -> Mendoza  naive 10.60 degrees | haversine    986 km | km per 'degree'  93.0
Buenos Aires -> Ushuaia  naive 22.50 degrees | haversine   2373 km | km per 'degree' 105.5


The "km per degree" column is not constant: there is no single factor that turns the naive number into kilometres. The reason is in the next cell.

In [5]:
def km_per_degree_of_longitude(lat):
    return 2 * math.pi * EARTH_RADIUS_KM * math.cos(math.radians(lat)) / 360


for name in ["Salta", "Buenos Aires", "Ushuaia"]:
    lat = cities[name][1]
    print(f"{name:<13} 1° of longitude = {km_per_degree_of_longitude(lat):5.1f} km   (1° of latitude is always ~111.2 km)")

Salta         1° of longitude = 101.0 km   (1° of latitude is always ~111.2 km)
Buenos Aires  1° of longitude =  91.5 km   (1° of latitude is always ~111.2 km)
Ushuaia       1° of longitude =  64.1 km   (1° of latitude is always ~111.2 km)


## Takeaways

- Coordinates are `(lon, lat)` in GIS libraries. Swapping them is the most common bug.
- Distances computed on raw degrees are not distances: one degree of longitude is ~102 km in Salta and ~64 km in Ushuaia.
- For distances, use a formula on the sphere (haversine) or the ellipsoid (`pyproj.Geod`, notebook 03), or project to a CRS in metres.

## Exercises

1. Add Rosario `(-60.6393, -32.9468)` to `cities` and print its distance to Buenos Aires.
2. Write `hemisphere(lon, lat)` that returns e.g. `"S/W"`.
3. Write `is_valid(lon, lat)` that rejects longitudes outside [-180, 180] and latitudes outside [-90, 90].